<a href="https://colab.research.google.com/github/jhanvib0249-cell/Unknown_UAVs/blob/main/Copy_of_UAV_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Code to train the model YOLOv8

In [ ]:
# 1. Install required packages in Colab environment
!pip install ultralytics kagglehub pyyaml

import os
import glob
import yaml
import kagglehub
from ultralytics import YOLO

def train_drone_detector_fast():
    print("=== STEP 1: Setting up Dataset ===")
    dataset_path = kagglehub.dataset_download("muki2003/yolo-drone-detection-dataset")
    print(f"Dataset path: {dataset_path}")

    # Find the original data.yaml file
    yaml_files = glob.glob(os.path.join(dataset_path, "**", "*.yaml"), recursive=True)
    if not yaml_files:
        raise FileNotFoundError("data.yaml file not found in dataset folder.")

    orig_yaml_path = yaml_files[0]
    dataset_dir = os.path.dirname(orig_yaml_path)

    # 1. Read from the read-only dataset YAML
    with open(orig_yaml_path, "r") as f:
        yaml_content = yaml.safe_load(f)

    # 2. Update paths inside the dictionary
    yaml_content["path"] = dataset_dir.replace("\\", "/")
    yaml_content["train"] = "train"
    yaml_content["val"] = "valid"

    # 3. FIX: Save as a NEW file in the current writable working directory
    custom_yaml_path = os.path.join(os.getcwd(), "custom_data.yaml")
    with open(custom_yaml_path, "w") as f:
        yaml.dump(yaml_content, f, default_flow_style=False)

    print(f"Created writable configuration file at: {custom_yaml_path}")

    print("\n=== STEP 2: Initializing Model ===")
    last_checkpoint = "runs/detect/uav_detector/weights/last.pt"

    # Auto-resume check if last.pt exists
    if os.path.exists(last_checkpoint):
        print(f"Found interrupted checkpoint at '{last_checkpoint}'. Resuming...")
        model = YOLO(last_checkpoint)
        resume_flag = True
    else:
        print("Starting fresh training with 3 epochs...")
        model = YOLO("yolov8n.pt")
        resume_flag = False

    print("\n=== STEP 3: Starting Training ===")
    results = model.train(
        data=custom_yaml_path, # Pass our new custom_data.yaml
        epochs=3,              # Fast 3-epoch run
        imgsz=640,
        batch=16,
        name="uav_detector",
        exist_ok=True,
        resume=resume_flag
    )

    print("\nTraining complete!")
    print("Your trained weights are saved at: runs/detect/uav_detector/weights/best.pt")

    # If running on Colab, trigger download
    try:
        from google.colab import files
        best_weights = "runs/detect/uav_detector/weights/best.pt"
        if os.path.exists(best_weights):
            print("Downloading best.pt to your computer...")
            files.download(best_weights)
    except ImportError:
        pass

if __name__ == "__main__":
    train_drone_detector_fast()

Code for prediction of an unknown UAV by uploading an image from the system

In [ ]:
# 1. Install required packages in Colab environment
#!pip install ultralytics kagglehub pyyaml

from google.colab import files
from ultralytics import YOLO
import cv2
from google.colab.patches import cv2_imshow

# 1. Prompt file upload dialog in Colab
print("Upload a test UAV image from your computer:")
uploaded = files.upload()

# 2. Load trained model
model = YOLO("runs/detect/uav_detector/weights/best.pt")

# 3. Run prediction on uploaded file(s)
for file_name in uploaded.keys():
    print(f"\nProcessing {file_name}...")
    results = model.predict(source=file_name, conf=0.25)

    for result in results:
        annotated_frame = result.plot()
        cv2_imshow(annotated_frame)  # Show in notebook

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 989.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Upload a test UAV image from your computer:


Saving Drone Technology _ How It’s Transforming Key Industries.jpg to Drone Technology _ How It’s Transforming Key Industries.jpg


FileNotFoundError: [Errno 2] No such file or directory: 'runs/detect/uav_detector/weights/best.pt'

Code to train and predict the UAV on an UI

In [ ]:
!pip install ultralytics kagglehub pyyaml gradio opencv-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.3 MB/s eta 0:00:00


In [ ]:
import os
import glob
import yaml
import kagglehub
import cv2
import gradio as gr
from ultralytics import YOLO

def train_drone_detector_fast():
    """Handles the dataset setup and model training."""
    try:
        print("=== STEP 1: Setting up Dataset ===")
        # Download dataset from Kaggle
        dataset_path = kagglehub.dataset_download("muki2003/yolo-drone-detection-dataset")

        # Find the original data.yaml file
        yaml_files = glob.glob(os.path.join(dataset_path, "**", "*.yaml"), recursive=True)
        if not yaml_files:
            return "Error: data.yaml file not found in dataset folder."

        orig_yaml_path = yaml_files[0]
        dataset_dir = os.path.dirname(orig_yaml_path)

        # Read from the read-only dataset YAML
        with open(orig_yaml_path, "r") as f:
            yaml_content = yaml.safe_load(f)

        # Update paths inside the dictionary
        yaml_content["path"] = dataset_dir.replace("\\", "/")
        yaml_content["train"] = "train"
        yaml_content["val"] = "valid"

        # Save as a NEW file in the current writable working directory
        custom_yaml_path = os.path.join(os.getcwd(), "custom_data.yaml")
        with open(custom_yaml_path, "w") as f:
            yaml.dump(yaml_content, f, default_flow_style=False)

        # Initializing Model
        last_checkpoint = "runs/detect/uav_detector/weights/last.pt"
        if os.path.exists(last_checkpoint):
            model = YOLO(last_checkpoint)
            resume_flag = True
        else:
            model = YOLO("yolov8n.pt")
            resume_flag = False

        # Starting Training
        model.train(
            data=custom_yaml_path,
            epochs=3,
            imgsz=640,
            batch=16,
            name="uav_detector",
            exist_ok=True,
            resume=resume_flag
        )
        return "Training complete! Weights are saved at runs/detect/uav_detector/weights/best.pt"

    except Exception as e:
        return f"Training failed: {str(e)}"

def detect_uav(image):
    """Handles inference on user-uploaded images."""
    if image is None:
        return None, "Error: Please upload an image first."

    model_path = "runs/detect/uav_detector/weights/best.pt"

    if not os.path.exists(model_path):
        return image, "Error: Trained model not found. Please run the 'Train Model' tab first."

    try:
        # Load trained model
        model = YOLO(model_path)

        # Run prediction on uploaded file(s) with confidence 0.25
        results = model.predict(source=image, conf=0.25)

        for result in results:
            annotated_frame = result.plot()
            # Convert BGR to RGB for Gradio UI display compatibility
            annotated_frame_rgb = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)
            return annotated_frame_rgb, "Detection successful!"

        return image, "No predictions returned."

    except Exception as e:
        return image, f"Detection error: {str(e)}"

# === Gradio UI Layout ===
with gr.Blocks(theme=gr.themes.Soft()) as kshiti_ui:
    gr.Markdown("# 🚁 KSHITI UAV - YOLOv8 Detector Interface")
    gr.Markdown("Train your custom drone detection model and test it on new images instantly.")

    with gr.Tab("1. Detect UAV"):
        gr.Markdown("Upload an image to detect UAVs using the trained model.")
        with gr.Row():
            img_input = gr.Image(label="Upload Test Image")
            img_output = gr.Image(label="Detection Result")

        status_text = gr.Textbox(label="Status")
        detect_btn = gr.Button("Run Detection", variant="primary")

        detect_btn.click(fn=detect_uav, inputs=img_input, outputs=[img_output, status_text])

    with gr.Tab("2. Train Model"):
        gr.Markdown("Download the dataset and initiate a fast 3-epoch YOLO training session.")
        train_output = gr.Textbox(label="Training Status Log", lines=3)
        train_btn = gr.Button("Start Training", variant="primary")

        train_btn.click(fn=train_drone_detector_fast, inputs=None, outputs=train_output)

# Launch the UI in the Colab output cell
if __name__ == "__main__":
    kshiti_ui.launch(debug=True, share=True)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/tmp/ipykernel_746/2799033375.py:91: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as kshiti_ui:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2d56e3d236480cdf64.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


=== STEP 1: Setting up Dataset ===


100%|██████████| 359M/359M [00:02<00:00, 153MB/s]

Extracting files...


Ultralytics 8.4.108 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/custom_data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=uav_detector, nbs=64, nms=False, opset